[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block1.ipynb)

# Block 1: looking at data before you trust it

Nineteen minutes, together. Run each cell with **Shift and Enter**.

Nothing here has to be typed. Every answer is already written. Your job is to
read four or five lines and know what they did.

## 1. The six phones

This is the same table as the slide. Five columns, six rows. `price`,
`screen`, `battery` and `brand` are the features. `bought` is the label.

In [ ]:
import pandas as pd

rows = [
    (1200, 6.1, 4000, 'A', 'yes'),
    (3400, 6.7, 5000, 'B', 'yes'),
    (800, 5.5, 3000, 'C', 'no'),
    (2100, 6.4, 4500, 'A', 'yes'),
    (950, 6.0, 3200, 'C', 'no'),
    (4800, 6.8, 5200, 'B', 'no')
]
cols = ["price", "screen", "battery", "brand", "bought"]
phones = pd.DataFrame(rows, columns=cols)
phones

## 2. Three questions to ask any table

How big is it, what is in one column, and is anything missing.

In [ ]:
print("rows and columns:", phones.shape)
print("average price:   ", round(phones["price"].mean(), 1))
print("how many bought: ", phones["bought"].value_counts().to_dict())
print("missing anywhere:", phones.isna().sum().sum())

Nothing is missing from this one. Six rows is small enough to check by eye.

Real tables are not six rows.

## 3. Nine hundred phones

Now the same five columns on a bigger table. The table is made in the
notebook, so nothing has to be downloaded, and everybody in the room gets
exactly the same nine hundred rows. Most of the phones are cheap, and a
dearer phone tends to have a bigger battery.

In [ ]:
import numpy as np

rng = np.random.default_rng(20260804)
n = 900

# Most phones are cheap and a few are dear, and a dearer phone tends to have a
# bigger battery.
price = np.round(400 + rng.lognormal(np.log(1200), 0.6, n), -1)
big = pd.DataFrame({
    "price":   price,
    "screen":  np.round(np.clip(rng.normal(6.2, 0.35, n), 5.0, 7.0), 1),
    "battery": np.round(4000 + 1100 * np.log(price / 1600)
                        + rng.normal(0, 250, n)),
    "brand":   rng.choice(["A", "B", "C"], n),
    "bought":  np.where(rng.random(n) < 1 / (1 + price / 2000), "yes", "no"),
})

# Every battery, kept before any of them go missing. A real table never comes
# with this; this one was made here, so it can.
every_battery = big["battery"].copy()

# Punch the holes the chart on the slide showed, in the same columns and the
# same proportions, so this table is that table. A phone under 1500 is ten
# times as likely as a dearer one to be missing its battery.
for column, percent in (('price', 2), ('screen', 11), ('battery', 34), ('brand', 0), ('bought', 0)):
    if percent:
        chance = np.ones(n)
        if column == "battery":
            chance = np.where(price < 1500, 10.0, 1.0)
        holes = rng.choice(n, round(n * percent / 100), replace=False,
                           p=chance / chance.sum())
        big.loc[holes, column] = np.nan

big.shape

## 4. What is missing, counted

One line, and every number after it depends on the answer.

In [ ]:
big.isna().sum()

`battery` is missing 306 times out of 900, which is the 34 percent the chart on
the slide showed. That is one phone in three.

If you take the average battery now, you are taking the average of the phones
that happened to have one recorded, which answers a different question.

In [ ]:
print("average battery, where one is recorded:", round(big["battery"].mean(), 1))
print("average battery, every phone:          ", round(every_battery.mean(), 1))
print("rows with every column filled in:", len(big.dropna()), "of", len(big))

The table was made in this notebook, so `every_battery` still holds the
readings that went missing. A real table never comes with that column.

The two averages differ because the holes fall where the phones are cheap:
cheap phones lose their battery reading more often, and cheap phones have
smaller batteries. The phones that kept a reading are the dearer ones, so their
average comes out higher.

The last line is what dropping every incomplete row leaves. A row goes when
any one of its columns is empty, so a phone missing only its screen size loses
its price and its battery too.

## 5. Draw it

On the left, the batteries: every phone in grey, and in blue the phones that
still have a reading. The grey showing at the low end is what the holes took.
On the right, the share of each column that is empty.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))

bins = range(2600, 5801, 200)
ax[0].hist(every_battery, bins=bins, color="#b9bfc9", label="every phone")
ax[0].hist(big["battery"].dropna(), bins=bins, color="#1e3a5f",
           label="battery recorded")
ax[0].set_xlabel("battery")
ax[0].set_ylabel("how many phones")
ax[0].legend(frameon=False)

missing = 100 * big.isna().sum() / len(big)
colours = ["#b45309" if c == "battery" else "#1e3a5f" for c in missing.index]
ax[1].barh(missing.index, missing.values, color=colours)
for row, share in enumerate(missing.values):
    ax[1].text(share + 0.5, row, "%.0f%%" % share, va="center")
ax[1].set_xlim(0, 40)
ax[1].invert_yaxis()
ax[1].set_xlabel("percent of rows where the column is empty")

plt.tight_layout()
plt.show()

## 6. Your turn

Two questions. Try them, then run the next cell.

1. How many phones have a screen bigger than 6.5 inches?
2. What is the average battery of brand B?

In [ ]:
print("screens over 6.5:", (big["screen"] > 6.5).sum())
print("brand B battery: ", round(big[big["brand"] == "B"]["battery"].mean(), 1))

## What just happened

You loaded a table, counted its rows and columns, found what was missing, and
drew a picture of it. Nothing here made a prediction and nothing here is a
model.

That was deliberate. A prediction from a table nobody looked at is not worth
making.